# Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Current Registry

In [2]:
registry_df = pd.read_csv("current_rental_registrations_251001.csv")

In [3]:
registry_df = registry_df[["RegisteredAddress"]]


In [4]:
registry_df["formatted_address"] = registry_df["RegisteredAddress"].str.replace(r"\s+,", ",", regex=True)
registry_df["formatted_address"] = registry_df["formatted_address"].astype(str).str.upper().str.strip()
# Remove unit numbers before the first comma (e.g., " AVE 5," → " AVE,")
registry_df["formatted_address"] = registry_df["formatted_address"].str.replace(
    r"\s+\d+(?=,)", "", regex=True
).str.strip()


In [5]:
import re

def clean_units(address):
    # Match pattern: [street] [unit], [city], MA [ZIP]
    match = re.match(r"^(.*\b(?:ST|AV|AVE|RD|BLVD|PL|CT|DR|TER|WAY|LN|SQ|TE|CIR|PKWY|PLZ|HWY))\s+[A-Z0-9\-]+, (.+?, MA \d{5})$", address)
    if match:
        return f"{match.group(1)}, {match.group(2)}"
    return address

registry_df["formatted_address"] = registry_df["formatted_address"].apply(clean_units)


In [6]:
registry_df['formatted_address'].sample(30)

4201                   10 RIDGE ST, ROSLINDALE, MA 02131
24932             15 COMMONWEALTH CT, BRIGHTON, MA 02135
19812           370 CHESTNUT HILL AV, BRIGHTON, MA 02135
8446                  74 SYDNEY ST, DORCHESTER, MA 02125
28028               46-48 EDWIN ST, DORCHESTER, MA 02124
26783                  32 DARTMOUTH ST, BOSTON, MA 02116
29700               12 FISHER AV, MISSION HILL, MA 02120
33106                  40 HAYDN ST, ROSLINDALE, MA 02131
1339                  111 OCEAN ST, DORCHESTER, MA 02124
27418          371 DORCHESTER ST, SOUTH BOSTON, MA 02127
27967                69 EDGEWATER DR, MATTAPAN, MA 02126
15358             15 BOYNTON ST, JAMAICA PLAIN, MA 02130
13889             530-532 COLUMBUS AV, ROXBURY, MA 02118
23307           1633 COMMONWEALTH AV, BRIGHTON, MA 02135
31525               135 GRANITE AV, DORCHESTER, MA 02124
31726              41 GREENWICH ST, DORCHESTER, MA 02122
20191              177 WEBSTER ST, EAST BOSTON, MA 02128
24480              381 COMMONWE

# Assessor Dataset

In [7]:
assessment_df = pd.read_csv("fy2025-property-assessment-data_12_30_2024.csv", dtype={21: str}, low_memory=False)

In [8]:

assessment_df = assessment_df[["ST_NUM", "ST_NUM2","ST_NAME","ZIP_CODE","CITY"]]


In [9]:
def combine_street_numbers(row):
    try:
        st_num = str(int(float(row['ST_NUM']))) if pd.notnull(row['ST_NUM']) else ""
        st_num2 = str(int(float(row['ST_NUM2']))) if pd.notnull(row['ST_NUM2']) else ""
        return f"{st_num}-{st_num2}" if st_num and st_num2 else st_num
    except:
        return ""

assessment_df["ST_NUM_COMBINED"] = assessment_df.apply(combine_street_numbers, axis=1)
assessment_df["ZIP_CODE_CLEAN"] = assessment_df["ZIP_CODE"].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

assessment_df["ST_NAME_CLEAN"] = assessment_df["ST_NAME"].astype(str).str.replace(r'\bAVE\.', 'AV', regex=True)

assessment_df["formatted_address"] = (
    assessment_df["ST_NUM_COMBINED"].str.strip() + " " +
    assessment_df["ST_NAME_CLEAN"].astype(str).str.strip() + ", " +
    assessment_df["CITY"].astype(str).str.strip() + ", MA " +
    assessment_df["ZIP_CODE_CLEAN"]
).str.upper()
assessment_df["formatted_address"].dropna().unique()[:10]

array(['104 PUTNAM ST, EAST BOSTON, MA 02128',
       '197 LEXINGTON ST, EAST BOSTON, MA 02128',
       '199 LEXINGTON ST, EAST BOSTON, MA 02128',
       '201 LEXINGTON ST, EAST BOSTON, MA 02128',
       '203 LEXINGTON ST, EAST BOSTON, MA 02128',
       '205-207 LEXINGTON ST, EAST BOSTON, MA 02128',
       '209-211 LEXINGTON ST, EAST BOSTON, MA 02128',
       '213 LEXINGTON ST, EAST BOSTON, MA 02128',
       '215 LEXINGTON ST, EAST BOSTON, MA 02128',
       '217 LEXINGTON ST, EAST BOSTON, MA 02128'], dtype=object)

In [10]:
assessment_df[["ST_NUM", "ST_NUM2", "ST_NAME", "CITY", "ZIP_CODE", "formatted_address"]].sample(50)


,ST_NUM,ST_NUM2,ST_NAME,CITY,ZIP_CODE,formatted_address
99369,110.0,NaN,SAWYER AV,DORCHESTER,2125.0,"110 SAWYER AV, DORCHESTER, MA 02125"
68411,793.0,NaN,E THIRD ST,SOUTH BOSTON,2127.0,"793 E THIRD ST, SOUTH BOSTON, MA 02127"
156026,75.0,NaN,HEWLETT ST,ROSLINDALE,2131.0,"75 HEWLETT ST, ROSLINDALE, MA 02131"
93136,14.0,NaN,ANSON ST,JAMAICA PLAIN,2130.0,"14 ANSON ST, JAMAICA PLAIN, MA 02130"
35942,40.0,NaN,Montgomery ST,BOSTON,2116.0,"40 MONTGOMERY ST, BOSTON, MA 02116"
150297,6.0,NaN,HAYES RD,ROSLINDALE,2131.0,"6 HAYES RD, ROSLINDALE, MA 02131"
100812,312.0,NaN,SAVIN HILL AV,DORCHESTER,2125.0,"312 SAVIN HILL AV, DORCHESTER, MA 02125"
59641,190.0,NaN,W SIXTH ST,SOUTH BOSTON,2127.0,"190 W SIXTH ST, SOUTH BOSTON, MA 02127"
107172,150.0,NaN,Wellington Hill ST,MATTAPAN,2126.0,"150 WELLINGTON HILL ST, MATTAPAN, MA 02126"
10438,77.0,NaN,PEARL ST,CHARLESTOWN,2129.0,"77 PEARL ST, CHARLESTOWN, MA 02129"


# Current vs Assessor MinHash

In [11]:
# STEP 0: Required Libraries
from datasketch import MinHash, MinHashLSH
import pandas as pd
import re

# STEP 1: Tokenizer + MinHash generator
def tokenize(address, ngram=3):
    address = re.sub(r'[^\w\s]', '', str(address).upper())
    return set(address[i:i+ngram] for i in range(len(address) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# STEP 2: Extract house number
def extract_house_number(address):
    match = re.match(r"^(\d+)", str(address).strip())
    return int(match.group(1)) if match else None

# STEP 3: Safe MinHash with house number check
def safe_minhash_match(address, lsh_index):
    try:
        tokens = tokenize(address)
        input_num = extract_house_number(address)
        m = create_minhash(tokens)
        matches = lsh_index.query(m)
        for match_addr in matches:
            match_num = extract_house_number(match_addr)
            if input_num == match_num:
                return match_addr
        return None
    except:
        return None



In [12]:
# Make sure your registry address column is cleaned and ready
registered_addresses = registry_df["formatted_address"].dropna().unique().tolist()

# Build LSH index
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in registered_addresses:
    m = create_minhash(tokenize(addr))
    lsh.insert(addr, m)



In [13]:
assessment_df["minhash_strict"] = assessment_df["formatted_address"].apply(lambda x: safe_minhash_match(x, lsh))

# Filter unmatched = unregistered
unregistered_assessor_strict = assessment_df[assessment_df["minhash_strict"].isnull()]

In [14]:
matched_assessor = assessment_df[assessment_df["minhash_strict"].notnull()]
matched_assessor[["formatted_address", "minhash_strict"]].head(20)


,formatted_address,minhash_strict
1,"197 LEXINGTON ST, EAST BOSTON, MA 02128","197 LEXINGTON ST, EAST BOSTON, MA 02128"
2,"199 LEXINGTON ST, EAST BOSTON, MA 02128","199 LEXINGTON ST, EAST BOSTON, MA 02128"
4,"203 LEXINGTON ST, EAST BOSTON, MA 02128","203 LEXINGTON ST, EAST BOSTON, MA 02128"
5,"205-207 LEXINGTON ST, EAST BOSTON, MA 02128","205-207 LEXINGTON ST, EAST BOSTON, MA 02128"
6,"209-211 LEXINGTON ST, EAST BOSTON, MA 02128","209 LEXINGTON ST, EAST BOSTON, MA 02128"
7,"213 LEXINGTON ST, EAST BOSTON, MA 02128","213 LEXINGTON ST, EAST BOSTON, MA 02128"
8,"215 LEXINGTON ST, EAST BOSTON, MA 02128","215 BENNINGTON ST, EAST BOSTON, MA 02128"
9,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"
10,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"
11,"217 LEXINGTON ST, EAST BOSTON, MA 02128","217 LEXINGTON ST, EAST BOSTON, MA 02128"


In [15]:
total_assessor = assessment_df["formatted_address"].notna().sum()
unregistered_assessor_strict_count = len(unregistered_assessor_strict)
percent_unregistered_assessor = round((unregistered_assessor_strict_count / total_assessor) * 100, 2)
print("🏠 Assessor Properties")
print(f"Total formatted addresses: {total_assessor}")
print(f"Unregistered addresses: {unregistered_assessor_strict_count}")
print(f"Percentage unregistered: {percent_unregistered_assessor}%")


🏠 Assessor Properties
Total formatted addresses: 183442
Unregistered addresses: 127052
Percentage unregistered: 69.26%


# 311 Dataset

In [16]:
service_df = pd.read_csv("dff4d804-5031-443a-8409-8344efd0e5c8.csv", low_memory=False)

In [17]:
service_df = service_df[["location"]]


In [18]:
# Known city/neighborhood names to catch multi-word places like "SOUTH BOSTON", "JAMAICA PLAIN"
boston_neighborhoods = [
    "SOUTH BOSTON", "EAST BOSTON", "JAMAICA PLAIN", "MATTAPAN", "ROXBURY", 
    "BRIGHTON", "CHARLESTOWN", "HYDE PARK", "DORCHESTER", "WEST ROXBURY", 
    "ALLSTON", "ROSLINDALE", "BACK BAY", "FENWAY", "MISSION HILL", "NORTH END",
    "SOUTH END", "CHINATOWN"
]

def format_location(location):
    try:
        parts = location.strip().split()
        if len(parts) < 4:
            return location.upper()

        zip_code = parts[-1]
        state = parts[-2]

        # Try 2-word city names first
        possible_city = " ".join(parts[-4:-2]).upper()
        if possible_city in boston_neighborhoods:
            city = possible_city
            street = " ".join(parts[:-4])
        else:
            # Fall back to 1-word city names
            city = parts[-3].upper()
            street = " ".join(parts[:-3])

        return f"{street}, {city}, {state} {zip_code}".upper()
    except:
        return ""
service_df["formatted_address"] = service_df["location"].astype(str).apply(format_location)

# Replace AVE (or AVE.) with AV in 311 formatted addresses
service_df["formatted_address"] = service_df["formatted_address"].str.replace(
    r"\bAVE\.?\b", "AV", regex=True
)


service_df["formatted_address"].sample(30)

14850               272 RESERVATION RD, HYDE PARK, MA 02136
118516                 7 GRANDVIEW ST, ROSLINDALE, MA 02131
22451           501-509 WASHINGTON ST, DORCHESTER, MA 02124
233937    INTERSECTION OF SOLDIERS FIELD PL & SOLDIERS F...
275895                1310 WASHINGTON ST, ROXBURY, MA 02118
28030                 5A-5 SEAVER ST, EAST BOSTON, MA 02128
78358                   286 WALNUT AV, DORCHESTER, MA 02121
72706                    100-110 STATE ST, BOSTON, MA 02109
234463                  101-103 HUDSON ST, BOSTON, MA 02111
180181                381 COMMONWEALTH AV, BOSTON, MA 02215
116862              1576 TREMONT ST, MISSION HILL, MA 02120
131843                    2 LEWIS ST, EAST BOSTON, MA 02128
148233         16-20 CLIPPER SHIP LN, EAST BOSTON, MA 02128
134481             1762 COMMONWEALTH AV, BRIGHTON, MA 02135
200252               52 SAVIN HILL AV, DORCHESTER, MA 02125
135638                        16 MINER ST, BOSTON, MA 02215
30432                   10 EMMONS ST, EA

# Current Vs 311 MinHash

In [19]:
# STEP 0: Required Libraries
from datasketch import MinHash, MinHashLSH
import pandas as pd
import re

# STEP 1: Tokenizer + MinHash generator
def tokenize(address, ngram=3):
    address = re.sub(r'[^\w\s]', '', str(address).upper())
    return set(address[i:i+ngram] for i in range(len(address) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# STEP 2: Extract house number
def extract_house_number(address):
    match = re.match(r"^(\d+)", str(address).strip())
    return int(match.group(1)) if match else None

# STEP 3: Safe MinHash with house number check
def safe_minhash_match(address, lsh_index):
    try:
        tokens = tokenize(address)
        input_num = extract_house_number(address)
        m = create_minhash(tokens)
        matches = lsh_index.query(m)
        for match_addr in matches:
            match_num = extract_house_number(match_addr)
            if input_num == match_num:
                return match_addr
        return None
    except:
        return None


In [20]:
# Make sure your registry address column is cleaned and ready
registered_addresses = registry_df["formatted_address"].dropna().unique().tolist()

# Build LSH index
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in registered_addresses:
    m = create_minhash(tokenize(addr))
    lsh.insert(addr, m)


In [21]:
# Apply strict MinHash match with house number check
service_df["minhash_strict"] = service_df["formatted_address"].apply(lambda x: safe_minhash_match(x, lsh))

# Filter unmatched = unregistered
unregistered_311_strict = service_df[service_df["minhash_strict"].isnull()]

matched_311 = service_df[service_df["minhash_strict"].notnull()]

# Show side-by-side comparison
matched_311[["formatted_address", "minhash_strict"]].head(20)


,formatted_address,minhash_strict
2,"416 BELGRADE AV, WEST ROXBURY, MA 02132","416 BELGRADE AV, WEST ROXBURY, MA 02132"
3,"3 SAINT CHARLES ST, BOSTON, MA 02116","3 SAINT CHARLES ST, BOSTON, MA 02116"
9,"4 HARTWELL ST, DORCHESTER, MA 02121","4-6 HARTWELL ST, DORCHESTER, MA 02121"
10,"32 FAIRFIELD ST, BOSTON, MA 02116","32 FAIRFIELD ST, BOSTON, MA 02116"
17,"193 W NINTH ST, SOUTH BOSTON, MA 02127","193 W EIGHTH ST, SOUTH BOSTON, MA 02127"
19,"40 MORRIS ST, EAST BOSTON, MA 02128","40 MORRIS ST, EAST BOSTON, MA 02128"
20,"4 GILMER ST, MATTAPAN, MA 02126","4 GILMER ST, MATTAPAN, MA 02126"
25,"394 MERIDIAN ST, EAST BOSTON, MA 02128","394 MERIDIAN ST, EAST BOSTON, MA 02128"
27,"1200 WASHINGTON ST, ROXBURY, MA 02118","1200 WASHINGTON ST, ROXBURY, MA 02118"
29,"331 FANEUIL ST, BRIGHTON, MA 02135","331 FANEUIL ST, BRIGHTON, MA 02135"


In [22]:
total_311 = service_df["formatted_address"].notna().sum()
unregistered_311_strict_count = len(unregistered_311_strict)
percent_unregistered_311 = round((unregistered_311_strict_count / total_311) * 100, 2)
print("📋 311 Service Requests")
print(f"Total formatted addresses: {total_311}")
print(f"Unregistered addresses: {unregistered_311_strict_count}")
print(f"Percentage unregistered: {percent_unregistered_311}%\n")

📋 311 Service Requests
Total formatted addresses: 282836
Unregistered addresses: 220260
Percentage unregistered: 77.88%



# Unregistered List Concat

In [24]:
# Recombine both full rows
combined_unregistered_df = pd.concat([unregistered_311_strict, unregistered_assessor_strict], ignore_index=True)

# Drop duplicate addresses (but only if truly identical)
combined_unregistered_df = combined_unregistered_df.drop_duplicates(subset=["formatted_address"])

print(f" Total unique unregistered addresses (combined): {len(combined_unregistered_df)}")


# Now your house numbers are preserved
print("\n Sample addresses:")
print(combined_unregistered_df["formatted_address"].head(10))


 Total unique unregistered addresses (combined): 105589

 Sample addresses:
0          160-162 LIVERPOOL ST, EAST BOSTON, MA 02128
1             1660 DORCHESTER AV, DORCHESTER, MA 02122
2                   47 TORREY ST, DORCHESTER, MA 02124
3                  11 CLAXTON ST, ROSLINDALE, MA 02131
4     252-254 S HUNTINGTON AV, JAMAICA PLAIN, MA 02130
5                  39 RAMSDELL AV, HYDE PARK, MA 02136
6             84 BUNKER HILL ST, CHARLESTOWN, MA 02129
7    INTERSECTION OF ROWLEY ST & WORRELL, ST, DORCH...
8                  99 SHANDON RD, DORCHESTER, MA 02124
9                  150 THIRD AV, CHARLESTOWN, MA 02129
Name: formatted_address, dtype: object


In [28]:
combined_unregistered_df = combined_unregistered_df["formatted_address"]
combined_unregistered_df.to_csv("combined_unregistered_addresses.csv", index=False)


# SAM_ID Match

In [29]:
import pandas as pd
from datasketch import MinHash, MinHashLSH
import re

# Load your datasets
unregistered_df = pd.read_csv("combined_unregistered_addresses.csv")
sam_df = pd.read_csv("live_street_address_management_sam_addresses.csv")

# Extract just house number + street name
def extract_street_only(address):
    try:
        return str(address).split(",")[0].strip().upper()
    except:
        return ""

unregistered_df["street_only"] = unregistered_df.iloc[:, 0].apply(extract_street_only)
sam_df["street_only"] = sam_df["FULL_ADDRESS"].astype(str).apply(extract_street_only)

# Tokenization and MinHash functions
def tokenize(text, ngram=3):
    text = re.sub(r'[^\w\s]', '', str(text).upper())
    return set(text[i:i+ngram] for i in range(len(text) - ngram + 1))

def create_minhash(tokens):
    m = MinHash(num_perm=128)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

# Build LSH index from SAM street-only addresses
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for addr in sam_df["street_only"].dropna().unique():
    lsh.insert(addr, create_minhash(tokenize(addr)))

# Match each unregistered street address
def match_street_only(text):
    try:
        m = create_minhash(tokenize(text))
        results = lsh.query(m)
        return results[0] if results else None
    except:
        return None

unregistered_df["minhash_match"] = unregistered_df["street_only"].apply(match_street_only)

# Merge matched SAM ID back in
matched_df = unregistered_df.merge(
    sam_df[["street_only", "SAM_ADDRESS_ID", "FULL_ADDRESS"]],
    left_on="minhash_match",
    right_on="street_only",
    how="left"
)

# Total unregistered addresses
total_unregistered = len(unregistered_df)

# Matched rows (those that got a SAM ID)
matched_with_sam = matched_df["SAM_ADDRESS_ID"].notna().sum()

# Calculate percentage
percent_matched = round((matched_with_sam / total_unregistered) * 100, 2)

# Print results
print("📊 SAM ID Matching Summary")
print(f"- Total unregistered addresses: {total_unregistered}")
print(f"- Matched with SAM ID: {matched_with_sam}")
print(f"- Percentage matched: {percent_matched}%")



/var/folders/4h/gm1b6b155z73jqkthlx4bgv00000gn/T/ipykernel_75200/3324297159.py:7: DtypeWarning: Columns (6,7,15,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  sam_df = pd.read_csv("live_street_address_management_sam_addresses.csv")


📊 SAM ID Matching Summary
- Total unregistered addresses: 105588
- Matched with SAM ID: 91124
- Percentage matched: 86.3%


In [30]:
matched_df.to_csv("unregistered_with_sam_ids.csv", index=False)
print("✅ Done! Output saved to 'unregistered_with_sam_ids.csv'")

✅ Done! Output saved to 'unregistered_with_sam_ids.csv'
